# Exploratory Colab experiment

> Cleaned archive of the original graduation-project notebook. For new leakage-aware runs, use the reusable pipeline under src/ and scripts/.


In [ ]:
# Hücre 1: Gerekli Kütüphaneler ve Drive Bağlantısı
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet101, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc



In [ ]:
# Hücre 2: Sabitler ve Yol Tanımları
IMG_SIZE         = 299
BATCH_SIZE       = 32
INITIAL_LR       = 1e-4
EPOCHS_FE        = 20
EPOCHS_FT        = 80
FE_CHECKPOINT    = 'artifacts/best_resnet101_fe.h5'
FT_CHECKPOINT    = 'artifacts/best_resnet101_ft.h5'

BASE_DIR         = 'data/split'
TRAIN_DIR        = os.path.join(BASE_DIR, 'train')
TEST_DIR         = os.path.join(BASE_DIR, 'test')


In [ ]:
# Hücre 3: Data Augmentation & Generator’lar
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.15,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
# Hücre 4: Class Weights Hesapla
classes     = np.unique(train_gen.classes)
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_gen.classes
)
class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)


In [ ]:
# Hücre 5: ResNet101 + Yeni Head Tanımı
base = ResNet101(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
x = base.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
preds = Dense(train_gen.num_classes, activation='softmax')(x)

model = Model(inputs=base.input, outputs=preds)
model.summary()


In [ ]:
# Hücre 6: Feature Extraction Aşaması
for layer in base.layers:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fe_ckpt = ModelCheckpoint(FE_CHECKPOINT, monitor='val_accuracy',
                          save_best_only=True, verbose=1)
fe_rlrp = ReduceLROnPlateau(monitor='val_loss',
                            factor=0.5, patience=3,
                            min_lr=1e-7, verbose=1)

history_fe = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FE,
    callbacks=[fe_ckpt, fe_rlrp],
    class_weight=class_weights
)


In [ ]:
# Hücre 7: Fine-Tuning Aşaması
# Son 75 katmanı aç
for layer in base.layers[:-75]:
    layer.trainable = False
for layer in base.layers[-75:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR * 0.05),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

ft_ckpt = ModelCheckpoint(FT_CHECKPOINT, monitor='val_accuracy',
                          save_best_only=True, verbose=1)
ft_rlrp = ReduceLROnPlateau(monitor='val_loss',
                            factor=0.5, patience=2,
                            min_lr=1e-6, verbose=1)

history_ft = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FE + EPOCHS_FT,
    initial_epoch=EPOCHS_FE,
    callbacks=[ft_ckpt, ft_rlrp],
    class_weight=class_weights
)


In [ ]:
# Hücre 8: Eğitim Geçmişini Görselleştir
def plot_history(h, title):
    epochs = range(1, len(h.history['accuracy'])+1)
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(epochs, h.history['accuracy'], label='Train Acc')
    plt.plot(epochs, h.history['val_accuracy'], label='Val Acc')
    plt.title('Accuracy ' + title)
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(epochs, h.history['loss'], label='Train Loss')
    plt.plot(epochs, h.history['val_loss'], label='Val Loss')
    plt.title('Loss ' + title)
    plt.legend()
    plt.show()

plot_history(history_fe, ' (Feature Extraction)')
plot_history(history_ft, ' (Fine Tuning)')


In [ ]:
# Hücre 9: Test Seti Değerlendirmesi
# En iyi ağırlıkları yükle
model.load_weights(FT_CHECKPOINT)

# Tahminler
Y_pred = model.predict(test_gen)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_gen.classes
labels = list(test_gen.class_indices.keys())

# Confusion Matrix & Classification Report
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_true, y_pred, target_names=labels))


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Gerçek ve tahmin edilen olasılıkları al
y_true = test_gen.classes
Y_pred = model.predict(test_gen)

# Sınıf sayısını ve etiketleri al
n_classes = len(labels)

# y_true'u one-hot vektöre çevir
y_true_bin = label_binarize(y_true, classes=range(n_classes))

# ROC eğrileri ve AUC değerleri
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], Y_pred[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# ROC eğrisini çiz
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f'ROC curve (Class {labels[i]}) AUC = {roc_auc[i]:.2f}')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curves')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
# Gerekli kütüphaneler
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize

# 1. En iyi ağırlıkları yükle
model.load_weights(FT_CHECKPOINT)  # ← En iyi fine-tune checkpoint dosyası

# 2. Test verisini tahmin et
Y_pred = model.predict(test_gen)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_gen.classes
labels = list(test_gen.class_indices.keys())
n_classes = len(labels)

# 3. Confusion Matrix ve Classification Report
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Confusion Matrix - ResNet101")
plt.show()

print("📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=labels))

# 4. ROC Eğrileri
# Y_true'yu one-hot olarak dönüştür
y_true_bin = label_binarize(y_true, classes=range(n_classes))

# ROC eğrisi hesapla
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], Y_pred[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# ROC eğrilerini çiz
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f"Class {labels[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - ResNet101')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
from tensorflow.keras.models import load_model

# 1) Fine-tuning sırasında kaydedilen en iyi modeli yükle
best_ckpt_101 = 'artifacts/best_resnet101_ft.h5'  # ← Burası senin ModelCheckpoint ile kaydettiğin dosya

model = load_model(best_ckpt_101)  # Modeli ağırlıklarla birlikte yükle

# 2) Tüm modeli (mimari + ağırlık) tek bir .h5 dosyasına kaydet
final_model_path = 'artifacts/resnet101_eniyi.h5'
model.save(final_model_path)

print(f"✅ En iyi ResNet101 modeli tek .h5 dosyasında kaydedildi:\n{final_model_path}")


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# 1) En iyi modeli yükle (.h5 olarak tam model)
best_ckpt_101 = 'artifacts/resnet101_eniyi.h5'
model = load_model(best_ckpt_101)

# 2) Son 75 katmanı eğitime aç
for layer in model.layers[:-75]:
    layer.trainable = False
for layer in model.layers[-75:]:
    layer.trainable = True

# 3) Compile et (biraz daha yüksek bir learning rate ile)
model.compile(
    optimizer=Adam(learning_rate=5e-6),  # önceki 1e-6 yerine daha etkili bir seviye
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4) Callback'ler (yeni checkpoint dosya adı ile)
NEW_CKPT = 'artifacts/resnet101_ft75_final_100to130.h5'
checkpoint = ModelCheckpoint(NEW_CKPT, monitor='val_accuracy', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

# 5) Eğitimi devam ettir (epoch 100 → 130)
history_ft75 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=130,
    initial_epoch=100,
    callbacks=[checkpoint, reduce_lr],
    class_weight=class_weights
)


In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np
import matplotlib.pyplot as plt

# 1) En iyi model yükleniyor
model = load_model('artifacts/resnet101_final_30epoch.h5')

# 2) Tahminler
Y_pred = model.predict(test_gen)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_gen.classes
labels = list(test_gen.class_indices.keys())
n_classes = len(labels)

# 3) Accuracy / Loss grafiklerini çiz (history_ft ve history_ft30 birleşmeli)
# Eğer ayrı ayrı kaydettiysen:
# with open('history_ft.pkl', 'rb') as f: history_ft = pickle.load(f)
# with open('history_ft30.pkl', 'rb') as f: history_ft30 = pickle.load(f)

# Örnek olarak son fine-tuning tarihini çiziyoruz:
try:
    plt.plot(history_ft30.history['accuracy'], label='Train Accuracy')
    plt.plot(history_ft30.history['val_accuracy'], label='Val Accuracy')
    plt.title("Accuracy over Epochs (Final 30 Epoch)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.plot(history_ft30.history['loss'], label='Train Loss')
    plt.plot(history_ft30.history['val_loss'], label='Val Loss')
    plt.title("Loss over Epochs (Final 30 Epoch)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
except:
    print("⚠️ history_ft30 değişkeni bellekte yoksa pickle'dan yüklemeyi düşün.")

# 4) Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Confusion Matrix - ResNet101 (130 Epoch)")
plt.show()

# 5) Classification Report
print("📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=labels))

# 6) ROC Eğrisi (Multi-Class)
y_true_bin = label_binarize(y_true, classes=range(n_classes))
fpr = dict(); tpr = dict(); roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], Y_pred[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10, 8))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f"Class {labels[i]} (AUC = {roc_auc[i]:.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.title("Multi-class ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc='lower right')
plt.grid()
plt.show()

# 7) Ek metrikler: Sensitivity (Recall), Specificity, Accuracy
print("\n🔎 Ek Metrikler (Class bazlı):")
for i in range(n_classes):
    TP = cm[i, i]
    FN = np.sum(cm[i, :]) - TP
    FP = np.sum(cm[:, i]) - TP
    TN = np.sum(cm) - TP - FN - FP

    sensitivity = TP / (TP + FN) if (TP + FN) != 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) != 0 else 0
    accuracy    = (TP + TN) / np.sum(cm)

    print(f"\n{labels[i]}:")
    print(f"  Sensitivity (Recall):  {sensitivity:.2f}")
    print(f"  Specificity:           {specificity:.2f}")
    print(f"  Accuracy:              {accuracy:.2f}")


In [ ]:
# 1) Eğitim tamamlanan modeli hem mimari hem ağırlıklarla birlikte kaydet (.h5)
model.save('artifacts/resnet101_final_130epoch.h5')
print("✅ Model başarıyla kaydedildi: resnet101_final_130epoch.h5")


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ---- 0) Girdi boyutunu belirle ----
IMG_SIZE = model.input_shape[1]  # Modelin giriş boyutundan otomatik al

# ---- 1) Test verisini yükle ----
TEST_DIR = 'data/split/test'  # ← kendi test klasörüne göre güncelle

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=1,
    class_mode='categorical',
    shuffle=False
)

# ---- 2) Sınıf isimlerini sıraya koy ----
target_names = [class_name for class_name, _ in
                sorted(test_generator.class_indices.items(), key=lambda x: x[1])]

# ---- 3) Rastgele test görseli seç ----
file_paths = test_generator.filepaths
random_index = random.randint(0, len(file_paths) - 1)
img_path = file_paths[random_index]

# ---- 4) Görüntüyü yükle ve normalize et ----
img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# ---- 5) Model ile tahmin yap ----
pred_probs = model.predict(img_array)[0]
predicted_idx = np.argmax(pred_probs)
predicted_class = target_names[predicted_idx]

true_idx = test_generator.classes[random_index]
true_class = target_names[true_idx]

# ---- 6) Sonuçları yazdır ----
print(f"\n🟢 Gerçek Etiket  : {true_class}")
print(f"🔵 Tahmin Edilen  : {predicted_class}\n")

print("🎯 Olasılıklar:")
for i, name in enumerate(target_names):
    print(f"  {name}: {pred_probs[i]:.2f}")

# ---- 7) Görseli göster ----
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f"Gerçek: {true_class} / Tahmin: {predicted_class}")
plt.axis('off')
plt.show()


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# --- 1) Yeni class_weight hesapla ---
y_train = train_gen.classes  # sınıf etiketleri (0, 1, 2)
class_weights_array = compute_class_weight(class_weight='balanced',
                                           classes=np.unique(y_train),
                                           y=y_train)
class_weights_updated = dict(zip(np.unique(y_train), class_weights_array))

print("✅ class_weights_updated:", class_weights_updated)

# --- 2) Katmanları ayarla ---
for layer in model.layers[:-75]:
    layer.trainable = False
for layer in model.layers[-75:]:
    layer.trainable = True

# --- 3) Modeli compile et ---
model.compile(
    optimizer=Adam(learning_rate=5e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# --- 4) Callback'ler ---
checkpoint = ModelCheckpoint(
    'artifacts/resnet101_ft75_classweightfix.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# --- 5) Eğitimi başlat ---
history_fix = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=150,
    initial_epoch=130,
    callbacks=[checkpoint, reduce_lr],
    class_weight=class_weights_updated
)


In [ ]:
# En güncel modeli farklı bir adla kaydet
model.save('artifacts/resnet101_ft75_classweightfix_150epoch.h5')
print("✅ Model farklı isimle kaydedildi: resnet101_ft75_classweightfix_150epoch.h5")
